# BÀI 1: KHẢO SÁT VÀ TIỀN XỬ LÝ DỮ LIỆU

* **Môn học:** Khai thác dữ liệu
* **Thực hiện:** Nguyễn Trường Duy, Lý Gia Hào

---

## 1. Bộ dữ liệu D1: Diabetes 130-US hospitals

Sử dụng dữ liệu y tế (20.000 bản ghi, 45 thuộc tính) để chuẩn bị cho bài toán Phân lớp (dự đoán nguy cơ nhập viện lại của bệnh nhân tiểu đường) và Phân cụm (phân nhóm đặc điểm bệnh nhân).

In [11]:
import pandas as pd
import numpy as np
import os

print("=== BÀI 1: TIỀN XỬ LÝ DỮ LIỆU D1 (DIABETES 130-US HOSPITALS) ===\n")

try:
    os.makedirs('data/raw', exist_ok=True)

    # Đọc file dữ liệu bạn vừa tải
    print("[1] Đang đọc file diabetic_data.csv...")
    df = pd.read_csv('diabetic_data.csv') # Sửa đường dẫn nếu file nằm ở nơi khác

    # Trong bộ y tế này, dữ liệu thiếu được ký hiệu là dấu '?'
    df.replace('?', np.nan, inplace=True)

    # Lấy mẫu 20.000 dòng để chạy nhanh và mượt (Thỏa mãn luật >= 10.000)
    df = df.sample(n=20000, random_state=42)

    # Loại bỏ ID bệnh nhân (không có giá trị dự đoán) và các cột thiếu quá 40% dữ liệu
    cols_to_drop = ['encounter_id', 'patient_nbr', 'weight', 'payer_code', 'medical_specialty']
    df.drop(columns=cols_to_drop, inplace=True)

    # Chuyển bài toán thành Phân lớp nhị phân: 1 (Có nhập viện lại), 0 (Không)
    df['readmitted'] = df['readmitted'].apply(lambda x: 0 if x == 'NO' else 1)

    print(f"\n[2] THÔNG TIN BỘ DỮ LIỆU:\n- Số dòng: {df.shape[0]}\n- Số cột: {df.shape[1]} (Đã trừ các cột rác)")

    # Xử lý Missing Values
    print("\n[3] Đang điền dữ liệu khuyết...")
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    cat_cols = df.select_dtypes(include=['object']).columns

    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())
    for col in cat_cols:
        df[col] = df[col].fillna('Unknown')

    # Lưu file sạch
    output_path = 'D1_Cleaned_Diabetes.csv'
    df.to_csv(output_path, index=False)
    print(f"\n[4] HOÀN TẤT! Dữ liệu sạch đã lưu tại: {output_path}")

except Exception as e:
    print(f"LỖI: {e}. Vui lòng đảm bảo file 'diabetic_data.csv' nằm cùng thư mục với code.")

=== BÀI 1: TIỀN XỬ LÝ DỮ LIỆU D1 (DIABETES 130-US HOSPITALS) ===

[1] Đang đọc file diabetic_data.csv...

[2] THÔNG TIN BỘ DỮ LIỆU:
- Số dòng: 20000
- Số cột: 45 (Đã trừ các cột rác)

[3] Đang điền dữ liệu khuyết...

[4] HOÀN TẤT! Dữ liệu sạch đã lưu tại: D1_Cleaned_Diabetes.csv


## 2. Bộ dữ liệu D2: Hotel Bookings

Sử dụng dữ liệu quản trị khách sạn (20.000 bản ghi, 32 thuộc tính) để chuẩn bị cho bài toán Phân lớp (dự đoán hành vi hủy phòng) và Phân cụm (phân khúc tập khách hàng dựa trên thói quen đặt phòng).

In [12]:
import pandas as pd
import numpy as np
import os

print("=== BÀI 1: KHẢO SÁT VÀ TIỀN XỬ LÝ DỮ LIỆU D2 (HOTEL BOOKINGS) ===\n")

try:
    os.makedirs('data/raw', exist_ok=True)

    # 1. Đọc dữ liệu
    file_path = 'hotel_bookings.csv'
    if not os.path.exists(file_path):
        file_path = 'hotel_bookings.csv'

    print(f"[1] Đang đọc dữ liệu từ '{file_path}'...")
    df2_raw = pd.read_csv(file_path)

    # 2. Khảo sát dữ liệu ban đầu
    print("\n[2] THÔNG TIN BỘ DỮ LIỆU GỐC:")
    print(f"- Số bản ghi: {df2_raw.shape[0]} dòng")
    print(f"- Số thuộc tính: {df2_raw.shape[1]} cột (Thỏa mãn luật >= 30)")
    print(f"- Phân phối biến mục tiêu (is_canceled):\n{df2_raw['is_canceled'].value_counts(normalize=True)*100}")

    # Lấy mẫu 20.000 dòng ngẫu nhiên để các bài sau chạy mượt mà trên laptop
    df2 = df2_raw.sample(n=20000, random_state=42).copy()

    # 3. Xử lý dữ liệu thiếu (Missing Values)
    print("\n[3] Đang xử lý dữ liệu bị khuyết (Missing Values)...")

    # Với dữ liệu khách sạn, khuyết 'agent' hoặc 'company' nghĩa là khách tự đặt (điền 0)
    # Khuyết 'children' thường là không có trẻ em đi kèm (điền 0)
    for col in ['children', 'agent', 'company']:
        if col in df2.columns:
            df2[col] = df2[col].fillna(0)

    # Khuyết quốc gia ('country') thì gán nhãn 'Unknown'
    if 'country' in df2.columns:
        df2['country'] = df2['country'].fillna('Unknown')

    # Quét lại toàn bộ để đảm bảo không bỏ sót bất kỳ ô trống nào
    num_cols = df2.select_dtypes(include=['float64', 'int64']).columns
    cat_cols = df2.select_dtypes(include=['object']).columns

    for col in num_cols:
        df2[col] = df2[col].fillna(df2[col].median())
    for col in cat_cols:
        df2[col] = df2[col].fillna('Unknown')

    print("- Đã làm sạch 100% dữ liệu thiếu!")

    # 4. Xuất file dữ liệu sạch
    output_path = 'D2_Cleaned_HotelBookings.csv'
    df2.to_csv(output_path, index=False)
    print(f"\n[4] HOÀN TẤT: Dữ liệu sạch đã được xuất thành công ra file '{output_path}'")

except FileNotFoundError:
    print("LỖI: Không tìm thấy file 'hotel_bookings.csv'. Vui lòng kiểm tra lại thư mục!")
except Exception as e:
    print(f"LỖI HỆ THỐNG: {e}")

=== BÀI 1: KHẢO SÁT VÀ TIỀN XỬ LÝ DỮ LIỆU D2 (HOTEL BOOKINGS) ===

[1] Đang đọc dữ liệu từ 'hotel_bookings.csv'...

[2] THÔNG TIN BỘ DỮ LIỆU GỐC:
- Số bản ghi: 119390 dòng
- Số thuộc tính: 32 cột (Thỏa mãn luật >= 30)
- Phân phối biến mục tiêu (is_canceled):
is_canceled
0    62.958372
1    37.041628
Name: proportion, dtype: float64

[3] Đang xử lý dữ liệu bị khuyết (Missing Values)...
- Đã làm sạch 100% dữ liệu thiếu!

[4] HOÀN TẤT: Dữ liệu sạch đã được xuất thành công ra file 'D2_Cleaned_HotelBookings.csv'


## 3. Bộ dữ liệu D3: NSL-KDD (Network Intrusion)

Sử dụng dữ liệu lưu lượng mạng (15.000 bản ghi, 43 thuộc tính) để chuẩn bị cho bài toán Khai phá luật kết hợp (nhận diện các quy luật và dấu hiệu của một cuộc tấn công mạng).

In [13]:
import pandas as pd
import os

print("=== BÀI 1: TIỀN XỬ LÝ VÀ CHUYỂN ĐỔI DỮ LIỆU D3 (NSL-KDD) ===\n")

try:
    # 1. Khai báo 43 tên cột chuẩn của hệ thống mạng NSL-KDD
    # (Vì file gốc .txt không có tên cột)
    columns = [
        "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
        "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
        "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
        "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login",
        "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate",
        "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
        "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
        "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
        "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate",
        "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "attack_class", "difficulty_level"
    ]

    # 2. Đọc trực tiếp file TXT (hàm read_csv của Pandas có thể đọc được file TXT)
    file_path = 'KDDTrain+.txt'
    if not os.path.exists(file_path):
        file_path = 'KDDTrain+.txt' # Dự phòng nếu file nằm cùng thư mục code

    print(f"[1] Đang đọc file '{file_path}' và gán tên cho 43 thuộc tính...")

    # Đọc dữ liệu, gán tên cột
    df3_raw = pd.read_csv(file_path, names=columns)

    # 3. Lấy mẫu 15.000 dòng để chạy thuật toán Apriori mượt mà
    df3 = df3_raw.sample(n=15000, random_state=42).copy()

    # 4. Gom nhóm biến mục tiêu:
    # Nếu là 'normal' thì gán là Bình thường, nếu khác 'normal' thì là Tấn công (Attack)
    df3['attack_class'] = df3['attack_class'].apply(lambda x: 'normal' if x == 'normal' else 'attack')

    print("\n[2] THÔNG TIN BỘ DỮ LIỆU:")
    print(f"- Số bản ghi: {df3.shape[0]} dòng")
    print(f"- Số thuộc tính: {df3.shape[1]} cột")

    # 5. Xuất ra file CSV sạch sẽ
    os.makedirs('data/raw', exist_ok=True)
    output_path = 'D3_Cleaned_NSLKDD.csv'
    df3.to_csv(output_path, index=False)

    print(f"\n[3] HOÀN TẤT: Dữ liệu đã được chuyển đổi sang định dạng CSV và lưu tại '{output_path}'")
    print("-> Bây giờ bạn đã có file CSV siêu chuẩn để chạy Luật kết hợp (Association Rules)!")

except FileNotFoundError:
    print("LỖI: Không tìm thấy file 'KDDTrain+.txt'. Bạn hãy kiểm tra lại xem đã tải và để đúng thư mục chưa nhé!")
except Exception as e:
    print(f"LỖI HỆ THỐNG: {e}")

=== BÀI 1: TIỀN XỬ LÝ VÀ CHUYỂN ĐỔI DỮ LIỆU D3 (NSL-KDD) ===

[1] Đang đọc file 'KDDTrain+.txt' và gán tên cho 43 thuộc tính...

[2] THÔNG TIN BỘ DỮ LIỆU:
- Số bản ghi: 15000 dòng
- Số thuộc tính: 43 cột

[3] HOÀN TẤT: Dữ liệu đã được chuyển đổi sang định dạng CSV và lưu tại 'D3_Cleaned_NSLKDD.csv'
-> Bây giờ bạn đã có file CSV siêu chuẩn để chạy Luật kết hợp (Association Rules)!
